## Datos de Startups (certificadas por ENISA)

Se combinará la información de extraida del panel de PowerBI de la fuente de ENISA con la información extraida de einforma a partir de los NIF de las startups

In [1]:
import pandas as pd
# diccionario de codigos cnae
cnae_df = pd.read_csv(
    "../data/silver/cnae_2025.csv", sep=";", encoding="utf-8", dtype=str
)
codigo_descripcion_dict = pd.Series(
    cnae_df.descripcion.values, index=cnae_df.codigo
).to_dict()
codigo_descripcion_subgrupo_dict = pd.Series(
    cnae_df.descripcion_subgrupo.values, index=cnae_df.codigo
).to_dict()
codigo_subgrupo_dict = pd.Series(
    cnae_df.subgrupo.values, index=cnae_df.codigo
).to_dict()
codigo_descripcion_grupo_dict = pd.Series(
    cnae_df.descripcion_grupo.values, index=cnae_df.codigo
).to_dict()
codigo_grupo_dict = pd.Series(cnae_df.grupo.values, index=cnae_df.codigo).to_dict()
codigo_descripcion_seccion_dict = pd.Series(
    cnae_df.descripcion_seccion.values, index=cnae_df.codigo
).to_dict()
codigo_seccion_dict = pd.Series(cnae_df.seccion.values, index=cnae_df.codigo).to_dict()

# diccionario de fechas de certificacion por ENISA
certificaciones_enisa_df = pd.read_csv("../data/bronze/ENISA/enisa_startups_certificadas.csv")
nif_fecha_certificacion_dict = pd.Series(
    certificaciones_enisa_df.fecha_certificacion.values, index=certificaciones_enisa_df.nif
).to_dict()
nif_fecha_estimada_descertificacion_dict = pd.Series(
    certificaciones_enisa_df.fecha_estimada_descertificacion.values,
    index=certificaciones_enisa_df.nif,
).to_dict()
nif_fecha_efecto_descertificacion_dict = pd.Series(
    certificaciones_enisa_df.fecha_efecto_descertificacion.values,
    index=certificaciones_enisa_df.nif,
).to_dict()

# diccionario de provincias a comunidades autonomas, municipios a provincias y codigos postales a municipios
provincias_ccaa = pd.read_csv(
    "../data/silver/ine_ccaa_y_provincias.csv", sep=";", encoding="utf-8", dtype=str
)
codigo_provincia_a_ccaa_dict = pd.Series(
    provincias_ccaa.comunidad_autonoma.values, index=provincias_ccaa.cpro
).to_dict()
codigo_provincia_a_codigo_ccaa_dict = pd.Series(
    provincias_ccaa.codauto.values, index=provincias_ccaa.cpro
).to_dict()
codigo_provincia_a_provincia_dict = pd.Series(
    provincias_ccaa.provincia.values, index=provincias_ccaa.cpro
).to_dict()
provincia_a_codigo_provincia_dict = pd.Series(
    provincias_ccaa.cpro.values, index=provincias_ccaa.provincia
).to_dict()
municipios_df = pd.read_csv(
    "../data/silver/municipios.csv", sep=",", encoding="utf-8", dtype=str
)
municipios_df['codigo_municipio'] = municipios_df['cpro'] + municipios_df['cmun']

codigo_municipio_a_municipio_dict = pd.Series(
    municipios_df.nombre.values, index=municipios_df.codigo_municipio
).to_dict()
municipio_a_codigo_municipio_dict = pd.Series(
    municipios_df.codigo_municipio.values, index=municipios_df.nombre
).to_dict()
codigo_municipio_a_provincia_dict = pd.Series(
    municipios_df.cpro.values, index=municipios_df.codigo_municipio
).to_dict()

codigos_postales_df = pd.read_csv(
    "../data/silver/caj_esp_012025.csv", sep=",", encoding="utf-8", dtype=str
)

cp_a_codigo_municipio_dict = pd.Series(
    codigos_postales_df.codigo_municipio.values, index=codigos_postales_df.codigo_postal
).to_dict()

Combinaremos los datos scrapeados de einforma con los datos de ENISA, el sistema cnae 2025 y la nomenclatura oficial de provincias y comunidades autonomas.

In [2]:
from src.geocoders.CorreosApi import CorreosApi
CorreosApi.set_config(log_level="ERROR")  # para evitar mensajes de log innecesarios

einforma_df = pd.read_csv(
    "../data/bronze/einforma/einforma_startups_certificadas.csv",
    sep=",",
    encoding="utf-8",
    dtype=str,
)
## limpieza y extracción de columnas #
einforma_df["cnae_2025"] = einforma_df["cnae_2025"].fillna(einforma_df["cnae_2009_2025"])  # porque cnae_2009-2025 contiene codigos validos de cnae_2025
einforma_df["codigo_cnae"] = einforma_df["cnae_2025"].str.partition(" - ")[0]  # extraccion del codigo cnae de la version más actual
einforma_df["codigo_cnae"] = einforma_df["codigo_cnae"].apply(lambda x: None if len(x)<4 else x)  # limpieza de codigos cnae erroneos
einforma_df["codigo_postal"] = einforma_df["localidad"].str.extract(r"^([0-9]{5})\s")
## adicion de provincias,comunidades autonomas
einforma_df["provincia"] = einforma_df["codigo_postal"].apply(
    lambda x: codigo_provincia_a_provincia_dict.get(str(x)[:2], None)
)  # se puede identificar la provincia con los 2 primeros digitos del codigo postal
einforma_df["cpro"] = einforma_df["provincia"].map(provincia_a_codigo_provincia_dict)
einforma_df["comunidad_autonoma"] = einforma_df["codigo_postal"].apply(
    lambda x: codigo_provincia_a_ccaa_dict.get(str(x)[:2], None)
)  # se puede identificar la comunidad autonoma con los 2 primeros digitos del codigo postal
einforma_df["ccaa"] = einforma_df['cpro'].map(codigo_provincia_a_codigo_ccaa_dict)
# traducir los codigos postales a codigos de municipio a partir del callejero censal, en caso contrario usar la api de Correos
einforma_df["cmun"] = einforma_df["codigo_postal"].apply(
    lambda x: cp_a_codigo_municipio_dict.get(str(x), CorreosApi.get_municipality_code_from_postal_code(str(x)))
)
einforma_df["municipio"] = einforma_df["cmun"].map(codigo_municipio_a_municipio_dict)
## adicion de descripciones cnae 2025 y niveles jerarquicos
einforma_df["descripcion_cnae"] = einforma_df["codigo_cnae"].map(codigo_descripcion_dict)
einforma_df["subgrupo_cnae"] = einforma_df["codigo_cnae"].map(codigo_subgrupo_dict)
einforma_df["descripcion_subgrupo_cnae"] = einforma_df["codigo_cnae"].map(
    codigo_descripcion_subgrupo_dict
)
einforma_df["grupo_cnae"] = einforma_df["codigo_cnae"].map(codigo_grupo_dict)
einforma_df["descripcion_grupo_cnae"] = einforma_df["codigo_cnae"].map(
    codigo_descripcion_grupo_dict
)
einforma_df["seccion_cnae"] = einforma_df["codigo_cnae"].map(codigo_seccion_dict)
einforma_df["descripcion_seccion_cnae"] = einforma_df["codigo_cnae"].map(
    codigo_descripcion_seccion_dict
)
## adicion de fechas de certificacion
einforma_df["fecha_certificacion_enisa"] = pd.to_datetime(
    einforma_df["nif"].map(nif_fecha_certificacion_dict)
).dt.date

einforma_df["fecha_estimada_descertificacion_enisa"] = pd.to_datetime(
    einforma_df["nif"].map(nif_fecha_estimada_descertificacion_dict)
).dt.date

einforma_df["fecha_efecto_descertificacion_enisa"] = pd.to_datetime(
    einforma_df["nif"].map(nif_fecha_efecto_descertificacion_dict)
).dt.date

Usaremos los datos de Iberinform sobre einforma debido a que incluye el dato del año de constitución y otros datos adicionales como el tramo de capital social.

In [3]:
import pandas as pd
iberinform_df = pd.read_csv(
    "../data/bronze/iberinform/iberinform_startups_certificadas.csv",
    sep=",",
    encoding="utf-8",
    dtype=str,
)

In [4]:
# inputar nulos en tramo_facturacion
filtro = iberinform_df['tramo_facturacion'].str.len() < 2
iberinform_df.loc[filtro, 'tramo_facturacion'] = None

Vemos un caso curioso donde una de las empresas ha cambiado de NIF porque ha cambiado de Forma jurídica de SL(B) a SA(A), esto se puede confirmar a través de infonif https://infonif.economia3.com/ficha-empresa/bytetravel-sa

In [5]:
iberinform_df[~iberinform_df['nif'].isin(certificaciones_enisa_df['nif'])]

,nif,estado,denominacion,marcas_nombres_comerciales,forma_juridica,direccion,codigo_postal,municipio,provincia,web,cnae,objeto_social,sector_empresa,fecha_constitucion,tramo_facturacion,tramo_capital_social,evolucion_ventas,tamaño_empresa,empleados
1739,A42788984,ACTIVA,BYTETRAVEL SA,BYTETRAVEL SA,Sociedad Anónima,"AVENIDA VIA AUGUSTA, 15, 25 , EDIFICIO A 2. 08...",08174,Sant Cugat Del Valles,BARCELONA,NaN,7911 - Actividades de las agencias de viajes,"Actividades De Agencia De Viajes, Mayorista Y ...","Actividades De Agencias De Viajes, Operadores ...",25/01/2021,6.000.000 – 15.000.000€,+1.000.000€,Aumenta,Mediana Empresa,26 – 50


In [6]:
certificaciones_enisa_df[certificaciones_enisa_df['nif'] == 'B42788984']

,denominacion,comunidad_autonoma,nif,provincia,fecha_estimada_descertificacion,fecha_certificacion,fecha_efecto_descertificacion
695,ESTA TRAVEL SERVICES SL,Cataluña,B42788984,Barcelona,NaN,2024-04-23 09:00:00,2026-01-25


In [7]:
# actualizar el nif
certificaciones_enisa_df.loc[certificaciones_enisa_df['nif'] == 'B42788984', 'nif'] = 'A42788984'

Encontramos 2 casos de cnae faltantes. IberInform no es del todo infalible, se emplea el scraper de eInforma como fallback en estos casos.

In [8]:
filtro = iberinform_df['cnae'].str.len()<5
iberinform_df[filtro][['nif','denominacion','direccion','cnae']]

,nif,denominacion,direccion,cnae
1016,B22659536,CELEROSPACE SL,"CALLE DE PEDRO HEREDIA, 9, 0 B. 28028, MADRID ...",-
1900,B22500474,NUFISA SALUD SL,"AVENIDA DEL MEDITERRANEO, 2, EDIFICIO ZAUDIN B...",-


In [9]:
from src.scrapers.einforma_scraper import get_data_from_nif
def get_any_field(data:dict,fields=['cnae_2025,cnae_2009_2025']):
    result = None
    for field in fields:
        result = data.get(field, None)
        if result is not None:
            break
    return result
     
iberinform_df.loc[filtro, 'cnae'] = iberinform_df.loc[filtro, 'nif'].apply(lambda x: get_any_field(get_data_from_nif(x), fields=['cnae_2025', 'cnae_2009_2025']) if pd.notnull(x) else None)

2026-05-22 14:04:07,751 | undetected_chromedriver.patcher | INFO | patching driver executable /home/guoqing/.local/share/undetected_chromedriver/undetected_chromedriver
2026-05-22 14:04:11,038 | EinformaScraper | WARNING | Element not found in https://www.einforma.com/servlet/app/prod/ETIQUETA_EMPRESA/nif/B22659536 for xpath: //td[strong[contains(text(), 'CNAE 2009 - 2025:')]]/following-sibling::td[1].
2026-05-22 14:04:12,060 | undetected_chromedriver.patcher | INFO | patching driver executable /home/guoqing/.local/share/undetected_chromedriver/undetected_chromedriver
2026-05-22 14:04:15,442 | EinformaScraper | WARNING | Element not found in https://www.einforma.com/servlet/app/prod/ETIQUETA_EMPRESA/nif/B22500474 for xpath: //td[strong[contains(text(), 'CNAE 2009 - 2025:')]]/following-sibling::td[1].


In [10]:
from src.geocoders.CorreosApi import CorreosApi
CorreosApi.set_config(log_level="ERROR")

iberinform_df["codigo_cnae"] = iberinform_df["cnae"].apply(
    lambda x: str(x).split(" - ")[0] if pd.notnull(x) else None
)  # extraccion del codigo cnae de la version más actual

# adicion de provincias,comunidades autonomas
iberinform_df["provincia"] = iberinform_df["codigo_postal"].apply(
    lambda x: codigo_provincia_a_provincia_dict.get(str(x)[:2], None)
)  # se puede identificar la provincia con los 2 primeros digitos del codigo postal
iberinform_df["cpro"] = iberinform_df["provincia"].map(provincia_a_codigo_provincia_dict)
iberinform_df["comunidad_autonoma"] = iberinform_df["codigo_postal"].apply(
    lambda x: codigo_provincia_a_ccaa_dict.get(str(x)[:2], None)
)  # se puede identificar la comunidad autonoma con los 2 primeros digitos del codigo postal
iberinform_df["ccaa"] = iberinform_df['cpro'].map(codigo_provincia_a_codigo_ccaa_dict)

iberinform_df["cmun"] = iberinform_df["codigo_postal"].apply(
    lambda x: cp_a_codigo_municipio_dict.get(str(x), CorreosApi.get_municipality_code_from_postal_code(str(x)))
)
iberinform_df["municipio"] = iberinform_df["cmun"].map(codigo_municipio_a_municipio_dict)

# adicion de descripciones cnae 2025 y niveles jerarquicos
iberinform_df["descripcion_cnae"] = iberinform_df["codigo_cnae"].map(
    codigo_descripcion_dict)
iberinform_df["subgrupo_cnae"] = iberinform_df["codigo_cnae"].map(codigo_subgrupo_dict)
iberinform_df["descripcion_subgrupo_cnae"] = iberinform_df["codigo_cnae"].map(
    codigo_descripcion_subgrupo_dict
)
iberinform_df["grupo_cnae"] = iberinform_df["codigo_cnae"].map(codigo_grupo_dict)
iberinform_df["descripcion_grupo_cnae"] = iberinform_df["codigo_cnae"].map(
    codigo_descripcion_grupo_dict
)
iberinform_df["seccion_cnae"] = iberinform_df["codigo_cnae"].map(codigo_seccion_dict)
iberinform_df["descripcion_seccion_cnae"] = iberinform_df["codigo_cnae"].map(
    codigo_descripcion_seccion_dict
)
# formateo de fechas
iberinform_df["fecha_constitucion"] = pd.to_datetime(iberinform_df["fecha_constitucion"], format='%d/%m/%Y', errors='coerce'
).dt.date
# adicion de fechas de certificacion
iberinform_df["fecha_certificacion_enisa"] = pd.to_datetime(
    iberinform_df["nif"].map(nif_fecha_certificacion_dict)
).dt.date

iberinform_df["fecha_estimada_descertificacion_enisa"] = pd.to_datetime(
    iberinform_df["nif"].map(nif_fecha_estimada_descertificacion_dict)
).dt.date

iberinform_df["fecha_efecto_descertificacion_enisa"] = pd.to_datetime(
    iberinform_df["nif"].map(nif_fecha_efecto_descertificacion_dict)
).dt.date

imputar valores nulos de estado a ACTIVO

In [ ]:
iberinform_df.fillna({'estado': 'ACTIVA'}, inplace=True)

,nif,estado,denominacion,marcas_nombres_comerciales,forma_juridica,direccion,codigo_postal,municipio,provincia,web,...,descripcion_cnae,subgrupo_cnae,descripcion_subgrupo_cnae,grupo_cnae,descripcion_grupo_cnae,seccion_cnae,descripcion_seccion_cnae,fecha_certificacion_enisa,fecha_estimada_descertificacion_enisa,fecha_efecto_descertificacion_enisa
0,B01827682,ACTIVA,EDUTECH COMMUNITY SL,EDUTECH COMMUNITY SL,Sociedad Limitada,"AVENIDA JAUME I, 95, P. 1. 08226, TERRASSA (BA...",08226,Terrassa,Barcelona,NaN,...,Otra educación n.c.o.p.,855,Otra educación,85,Educación,Q,EDUCACIÓN,2024-04-16,NaT,2025-12-02
1,B13964622,ACTIVA,DISRUPTIVE TEMPLE SL,DISRUPTIVE TEMPLE SL,Sociedad Limitada,"CARRETERA NACIONAL II TORRE D ARA, S/N, PL 0 B...",08302,Mataró,Barcelona,NaN,...,Comercio al por mayor no especializado,469,Comercio al por mayor no especializado,46,Comercio al por mayor,G,COMERCIO AL POR MAYOR Y AL POR MENOR,2026-02-25,2028-07-04,NaT
2,B16811408,ACTIVA,RHYDE.CO SL,RHYDE.CO SL,Sociedad Limitada,"CALLE DE DIAZ DEL CASTILLO, 12. 28823, COSLADA...",28823,Coslada,Madrid,https://es.rhyde.co,...,Reparación y mantenimiento de efectos personal...,952,Reparación y mantenimiento de efectos personal...,95,"Reparación y mantenimiento de ordenadores, art...",T,OTROS SERVICIOS,2024-04-23,2026-09-07,NaT
3,B13694161,ACTIVA,VASOMALY SL,VASOMALY SL,Sociedad Limitada,"AVENIDA FONTES, 15, 2 D. 30700, TORRE PACHECO ...",30700,Torre-Pacheco,Murcia,NaN,...,Investigación y desarrollo experimental en cie...,721,Investigación y desarrollo experimental en cie...,72,Investigación y desarrollo,N,"ACTIVIDADES PROFESIONALES, CIENTÍFICAS Y TÉCNICAS",2025-04-01,2030-05-03,NaT
4,B10964625,ACTIVA,EASY AND SECURE DIGITAL TRANSACTIONS SL,EASY AND SECURE DIGITAL TRANSACTIONS SL,Sociedad Limitada,"CALLE MAYOR, 16, B 02260. 02260, FUENTEALBILLA...",02260,Fuentealbilla,Albacete,NaN,...,Otros servicios relacionados con las tecnologí...,629,Otros servicios relacionados con las tecnologí...,62,"Programación, consultoría y otras actividades ...",K,"TELECOMUNICACIONES, PROGRAMACIÓN INFORMÁTICA, ...",2024-01-10,2027-08-17,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2157,B23835796,ACTIVA,PATRIMONIO GASTRONOMICO SL,PATRIMONIO GASTRONOMICO SL,Sociedad Limitada,"CALLE NEBRIJA, 5, PISO 1º PUERTA 3. 28801, ALC...",28801,Alcalá de Henares,Madrid,NaN,...,"Todas las demás actividades profesionales, cie...",749,"Otras actividades profesionales, científicas y...",74,"Otras actividades profesionales, científicas y...",N,"ACTIVIDADES PROFESIONALES, CIENTÍFICAS Y TÉCNICAS",2025-12-03,2030-09-25,NaT
2158,B56866106,ACTIVA,MATERIALS INNOVATION CONSULTING ASSOCIATES SL,MATERIALS INNOVATION CONSULTING ASSOCIATES SL,Sociedad Limitada,"CALLE ANTONIO BAENA, 18. 28035, MADRID (MADRID...",28035,Madrid,Madrid,NaN,...,Otras actividades de apoyo a las empresas n.c....,829,Otras actividades de apoyo a las empresas n.c....,82,Actividades administrativas de oficina y otras...,O,ACTIVIDADES ADMINISTRATIVAS Y SERVICIOS AUXILI...,2024-10-15,2029-01-08,NaT
2159,B70698428,ACTIVA,GLOTEGY SL,GLOTEGY SL,Sociedad Limitada,"CALLE PRINCIPE DE VERGARA, 120, ESC. 1 6 B. 28...",28002,Madrid,Madrid,NaN,...,Otras actividades de apoyo a las empresas n.c....,829,Otras actividades de apoyo a las empresas n.c....,82,Actividades administrativas de oficina y otras...,O,ACTIVIDADES ADMINISTRATIVAS Y SERVICIOS AUXILI...,2025-01-28,2029-04-25,NaT
2160,B16692733,ACTIVA,AUTOMATED INVESTMENT SOLUTIONS SL,AUTOMATED INVESTMENT SOLUTIONS SL,Sociedad Limitada,"AVENIDA GREGORIO PECES BARBA, 1. 28919, LEGANE...",28919,Leganés,Madrid,NaN,...,Actividades de programación informática,621,Actividades de programación informática,62,"Programación, consultoría y otras actividades ...",K,"TELECOMUNICACIONES, PROGRAMACIÓN INFORMÁTICA, ...",2025-06-24,2026-07-13,NaT


In [ ]:
iberinform_df['estado'].isna().sum()

np.int64(0)

Eliminar empresas sin datos de constitución ni codigo postal

In [25]:
print(iberinform_df.query("fecha_constitucion.isna() or codigo_postal.isna()")['nif'].count())
iberinform_df.dropna(subset=['fecha_constitucion', 'codigo_postal'], how='any', inplace=True)

2


Busqueda de códigos postales incorrectos a través de municipios nulos, se ha encontrado que 03080 no existe, el correcto seria 03690.
Como la empresa se encuentra dentro del período a estudiar se ecorrige en lugar de borrarlo.

In [28]:
iberinform_df.query('municipio.isna()')[['nif','denominacion','direccion','codigo_postal','provincia','fecha_constitucion']]

,nif,denominacion,direccion,codigo_postal,provincia,fecha_constitucion
978,B10806487,ADVANCED BUSINESS DIGITALIZATION SL,CALLE DE AL PQUE CIENTIFICO DEL CAMPUS DE LA U...,03080,Alicante/Alacant,2022-08-26


In [30]:
index = iberinform_df.query("nif == 'B10806487'").index
iberinform_df.loc[index, 'codigo_postal'] = '03690'
iberinform_df.loc[index, 'cmun'] = iberinform_df.loc[index, 'codigo_postal'].apply(lambda x: cp_a_codigo_municipio_dict.get(x, CorreosApi.get_municipality_code_from_postal_code(x)))
iberinform_df.loc[index, 'municipio'] = iberinform_df.loc[index, 'cmun'].apply(lambda x: codigo_municipio_a_municipio_dict.get(x, CorreosApi.get_municipality_name_from_postal_code(x)))

### Codigos postales a Latitud y Longitud

Se empleará la API de Correos para obtener la latitud y longitud a partir del código postal

Si las tareas de limpieza anterior ha sido correcto no debería de haber ninguna empresa sin municipio:

In [33]:

data_df = iberinform_df.copy()
data_df[data_df['municipio'].isna()][['nif','denominacion','direccion', 'codigo_postal','provincia']]

,nif,denominacion,direccion,codigo_postal,provincia


A continuación realizamos la geocodificación con la api de Correos. nos guardaremos el polígono del código postal y guardamos al archivo en la carpeta silver de data.

In [76]:
data_df[['latitud', 'longitud']] = data_df['codigo_postal'].apply(
    lambda x: pd.Series(CorreosApi.get_coords_from_postal_code(x, save_geojson=True))
)

In [77]:
data_df.latitud.isna().sum()

np.int64(0)

In [82]:
empresas_df = data_df.drop(columns=['cnae','sector_empresa']).copy()
empresas_df['nif'].count()

np.int64(2161)

In [ ]:
empresas_df.to_parquet("../data/silver/iberinform_startups_certificadas.parquet", index=False, compression='snappy', engine='pyarrow')

In [ ]:
# se recomienda comprobar errores con el registro mercantil, ya que a veces el CNAE no resulta el correcto.
import pandas as pd
empresas_df = pd.read_parquet("../data/silver/iberinform_startups_certificadas.parquet", engine='pyarrow')
registradores_df = pd.read_csv("../data/bronze/registradores/registradores_startups_certificadas.csv", sep=",", dtype=str)
nif_cnae_dict = registradores_df.set_index('nif')['cnae'].to_dict()
cnae_df = pd.read_csv("../data/silver/cnae_2025.csv", sep=";", dtype=str)
codigo_descripcion_dict = cnae_df.set_index('codigo')['descripcion'].to_dict()
grupo_descripcion_grupo_dict = cnae_df.set_index('grupo')['descripcion_grupo'].to_dict()
seccion_descripcion_seccion_dict = cnae_df.set_index('seccion')['descripcion_seccion'].to_dict()
grupo_seccion_dict = cnae_df.set_index('grupo')['seccion'].to_dict()
wrong_68_index = empresas_df.query('codigo_cnae.str.startswith("68")').index
wrong_56_index = empresas_df.query('codigo_cnae.str.startswith("56")').index
empresas_df.loc[wrong_68_index, 'codigo_cnae'] = empresas_df.loc[wrong_68_index, 'nif'].map(nif_cnae_dict)
empresas_df.loc[wrong_56_index, 'codigo_cnae'] = empresas_df.loc[wrong_56_index, 'nif'].map(nif_cnae_dict)
empresas_df["descripcion_cnae"] = empresas_df["codigo_cnae"].map(codigo_descripcion_dict)
empresas_df.loc[wrong_68_index, 'grupo_cnae'] = empresas_df.loc[wrong_68_index, 'codigo_cnae'].str[:2]
empresas_df.loc[wrong_56_index, 'grupo_cnae'] = empresas_df.loc[wrong_56_index, 'codigo_cnae'].str[:2]
empresas_df["descripcion_grupo_cnae"] = empresas_df["grupo_cnae"].map(grupo_descripcion_grupo_dict)
empresas_df.loc[wrong_68_index, 'seccion_cnae'] = empresas_df.loc[wrong_68_index, 'grupo_cnae'].map(grupo_seccion_dict)
empresas_df.loc[wrong_56_index, 'seccion_cnae'] = empresas_df.loc[wrong_56_index, 'grupo_cnae'].map(grupo_seccion_dict)
empresas_df["descripcion_seccion_cnae"] = empresas_df["seccion_cnae"].map(seccion_descripcion_seccion_dict)

empresas_df.to_parquet("../data/silver/iberinform_startups_certificadas.parquet", index=False, compression='snappy', engine='pyarrow')

# Datos Universidades del RECIDI

Principalemnte no interesa conocer las universidades públicas y privadas

In [ ]:
import pandas as pd

data_df = pd.read_excel(
    "../data/bronze/ciencia.gob.es/listauniversidades.xls",
    dtype=str
)

Vemos que la columna Modalidad es muy buen filtro para descartar datos de universidades que en realidad no aportan información real. Pues las universidades reales simepre tienen o Modalidad Presencial o No Presencial

In [ ]:
filtro_modalidad_nulo = data_df['Modalidad'].isna()

data_df[filtro_modalidad_nulo]['Universidad'].tolist()

In [ ]:
universidades_df = data_df.dropna(subset=['Modalidad']).copy().reset_index(drop=True)

In [ ]:
universidades_df.dtypes

Simplificaremso el tipo de universidad a pública y privada

In [ ]:
universidades_df['Tipo'].unique().tolist()

In [ ]:
universidades_df['Tipo'] = universidades_df['Tipo'].replace({
    'Privada (de la Iglesia Católica)': 'Privada',
    'Privada (concordataria)': 'Privada',
})
universidades_df['Tipo'].unique().tolist()

Obtenemos 98 universidades, 2 más que en el panel de PowerBi que aparece en la página del ministerio de ciencia, innovación y universidades. esto es debido a que el panel se construyó a partir de los datos del 30 de enero de 2025, pero según el RUCT del cual se basa el panel, ha habido 2 nuevas universidades reconocidas después del 30/01/2025. Por lo que a fecha de 23/01/2026 hay en vigor 98 universiades en el Estado español.

In [ ]:
universidades_df['Universidad'].count()

In [ ]:
universidades_df['Fecha publicación'] = universidades_df['Fecha publicación'].apply(lambda x: pd.to_datetime(x, errors='coerce').date() if pd.notnull(x) else None)
universidades_df[universidades_df['Fecha publicación'] > pd.to_datetime('2025-01-30').date()]

Nos interesa conocer los datos de Domicilio, Código postal y Municipio de cada Universidad para poder ubicarlo con coordenadas.

Las universidades con los datos faltantes son jústamente aquella con menos de 1 años de reconocimeinto oficial. Se descartarán porque se considera que el efecto que tendrán sobre el estudio será mínimo por el efecto de la antigüedad.

In [ ]:
filtro_domicilio_nulo = universidades_df['Domicilio'].isna()
filtro_codigo_postal_nulo = universidades_df['Código postal'].isna()
filtro_municipio_nulo = universidades_df['Municipio'].isna()

universidades_df[filtro_domicilio_nulo | filtro_codigo_postal_nulo | filtro_municipio_nulo][['Universidad','CIF','Domicilio','Código postal','Municipio']]

In [ ]:
universidades_df.dropna(subset=['Domicilio','Código postal','Municipio'], inplace=True)

A continuación extraeremos las columnas útiles y los cambiaremos a snake_case

In [ ]:
universidades_df.columns.tolist()

In [ ]:
from src.shared.utils import slugify, normalize

useful_columns = [
    'Universidad',
    'Código',
    'Acrónimo',
    'Tipo',
    'CIF',
    'Modalidad',
    'Domicilio',
    'Código postal',
    'Localidad',
    'Municipio',
    'Provincia',
    'Comunidad Autónoma',
    'URL',
    'Fecha publicación'
]

snake_case_columns = [slugify(normalize(col.lower())) for col in useful_columns]

cols_dict = dict(zip(useful_columns, snake_case_columns))

universidades_df.rename(columns=cols_dict, inplace=True)

universidades_df = universidades_df[[col for col in universidades_df.columns if col in snake_case_columns]].copy()



Vamos a enriquecer la información de las universidades con datos de su latitud y longitud.

In [ ]:
from src.shared.utils import normalize_address

In [ ]:
from geopy import ArcGIS

geocoder = ArcGIS(timeout=10)

def geocode_address(row):
    address = f"{row['universidad']}, {normalize_address(row['domicilio'])}, {row['codigo_postal']}, Spain"
    try:
        location = geocoder.geocode(address)
        if location:
            return pd.Series([location.latitude, location.longitude, location.address])
        else:
            print(f"Geocoding failed for address: {address}")
            return pd.Series([None, None, None])
    except Exception as e:
        print(f"Error geocoding address {address}: {e}")
        return pd.Series([None, None, None])

universidades_df[['latitud', 'longitud', 'direccion_geocoded']] = universidades_df.apply(geocode_address, axis=1)

In [ ]:
municipios_df = pd.read_csv(
    "../data/silver/municipios.csv", sep=",", encoding="utf-8", dtype=str
)
municipios_df['codigo_municipio'] = municipios_df['cpro'] + municipios_df['cmun']
municipio_cmun_dict = municipios_df.set_index('nombre')['codigo_municipio'].to_dict()
universidades_df['cmun'] = universidades_df['municipio'].map(municipio_cmun_dict)
universidades_df['cpro'] = universidades_df['cmun'].apply(lambda x: x[:2] if pd.notnull(x) else None)
ccaa_df = pd.read_csv(
    "../data/silver/ine_ccaa_y_provincias.csv", sep=";", encoding="utf-8", dtype=str
)
codigo_provincia_a_ccaa_dict = ccaa_df.set_index('cpro')['codauto'].to_dict()
universidades_df['ccaa'] = universidades_df['cpro'].map(codigo_provincia_a_ccaa_dict)
universidades_df

In [ ]:
universidades_df.to_csv(
    "../data/silver/universidades_arc_gis.csv", sep=";", index=False)

# Datos de Agrupaciones de Empresas Innovadoras

del archivo csv descargado de la pagina de de clusters.ipyme.org conteine en la fila 42 un cadena que contiene el mismo separador del csv por lo que fallará la lectura con pandas, se soluciona añadiendo doble comillas a la cadena de dirección con el separador.

In [ ]:
!ls ../data/bronze/clusters.ipyme.org/

El csv descargado del mapa de AEI de la pagina del ministerio de industria y turismo contiene en la linea 42 una dirección mal formateada que incluye el mismo caracter separador del csv ";", lo sustituiremos manualmente por una coma ','

In [ ]:
import pandas as pd
from src.shared.utils import normalize,slugify
data_df = pd.read_csv(
    "../data/bronze/clusters.ipyme.org/Agrupación Empresariales Innovadoras.csv", sep=";"
)

data_df.columns = [slugify(normalize(col.lower())) for col in data_df.columns]

Si nos fijamos en la columna de **direccion** del excel de AEIs, veremos que las direcciones pueden ser erroneas pues para la fila con el indice 3 y 4, ambas direcciones son la misma pero con formato distinto pero lo erroneo se encuentra en el código postal: 26001 y 26005, esto complica todavía más la geocodificación pues no bastará con normalizar las direcciones si no que habria que validar las direcciones y eso resulta casi imposible a nivel algoritmico. Por ello vamos a usar la api de google Maps que nos permitirá obtener un resultado muy acertado con datos que no son direcciones, como el nombre de la entidad o el dominio de su web.

Como se trata de 120 busquedas, resultarán gratuitas por encontrarse dentro de los créditos mensuales de GCP.

In [ ]:
data_df[['direccion']].head()

Para facilitar la búsqueda vamos a extraer de la dirección el codigo postal y provincia.

In [ ]:


data_df[['calle', 'codigo_postal', 'provincia']] = data_df['direccion'].str.extract(
    r'^(.*),\s*([0-9]{5})\s*,\s*(.*)$' # regex para filtrar el codigo postal y la provincia además de separarlo de la calle
)

In [ ]:
data_df[['calle', 'codigo_postal', 'provincia']].head()

In [ ]:
data_df[data_df['codigo_postal'].isna()]

Para usar la api de Google Maps debemos obtener una api key y guardarlo en el fichero *.env*


In [ ]:
from src.geocoders.google_maps import GoogleMaps as GMGeocoder
from src.shared.utils import extract_domain

def geocode_row(row):
    domain_available = True
    domain = extract_domain(row['web'])
    if '.' not in row['web']:
        print(f"Invalid web format: {row['web']}")
        
        if '@' not in row['correo_electronico']:
            print(f"Invalid email format: {row['correo_electronico']}")
            domain_available = False
        else:
            domain = extract_domain(row['correo_electronico'].split('@')[1])
    
    province = row['provincia']
    term = f"{domain}, {province}"
    if not domain_available:
        term = f"{row['direccion']}"
        domain = row['nombre']

    return GMGeocoder.geocode(term, save_json=True, filename=f"{domain}_geocoded.json")

In [ ]:
data_df['latitud'], data_df['longitud'], data_df['direccion_google_maps'] = zip(*data_df.apply(geocode_row, axis=1))

In [ ]:
data_df[['web','latitud','longitud', 'direccion_google_maps']]

In [ ]:
data_df[data_df['latitud'].isna()]

In [ ]:
aei_df = data_df[['nombre','sector','web','correo_electronico','latitud','longitud','codigo_postal','provincia','direccion','direccion_google_maps']].copy()

In [ ]:
aei_df['latitud'].isna().sum()

In [ ]:
aei_df.to_csv(
    "../data/silver/agrupaciones_empresariales_innovadoras.csv", sep=";", index=False)

# Datos Parques Científicos y Técnicos del RECIDI


Por sencillez vamos a usar la query del panel de powerBI incrustado en la página de los [centros RECIDI](https://www.ciencia.gob.es/Ministerio/Estadisticas/sicti/recidi.html).

Usaremos los mismo datos del panel con fecha de referencia a 30/01/2025.

Podríamos escrapear de la página de APTE pero esto resulta más sencillo.

In [1]:
from src.scrapers.powerbi.queries import recidi_science_and_technology_parks
from src.scrapers.powerbi.powerbi_parser import  pbi_json_to_df
data_df =  pbi_json_to_df(recidi_science_and_technology_parks())

In [2]:
from src.shared.utils import slugify, normalize
data_df.columns = [slugify(normalize(col.lower())) for col in data_df.columns]
data_df.columns

Index(['acronimo', 'nombre_entidad_mostrar', 'provincia', 'enlace_web'], dtype='str')

In [3]:
data_df.rename(columns={
    'nombre_entidad_mostrar':'nombre',
    'enlace_web':'web',
}, inplace=True)
data_df.columns

Index(['acronimo', 'nombre', 'provincia', 'web'], dtype='str')

In [ ]:
data_df.loc[data_df['acronimo'].str.len() == 0 | data_df['acronimo'].isna() ]

In [ ]:
from src.geocoders.google_maps import GoogleMaps as GMGeocoder

def geocode_row(row):
    acronimo = row['acronimo']
    if acronimo is None or len(acronimo.strip()) == 0:
        acronimo = row['nombre'].split('.')[0]
    term = f"{acronimo}, {row['provincia']}, Spain"
    return GMGeocoder.geocode(term, save_json=True, filename=f"{acronimo.replace(' ', '_')}_geocoded.json")
data_df['latitud'], data_df['longitud'], data_df['direccion_google_maps'] = zip(*data_df.apply(geocode_row, axis=1))

In [ ]:
data_df.head()

In [ ]:
data_df.to_csv(
    "../data/silver/parques_cientificos_tecnologicos_recidi.csv", sep=";", index=False)

# Centros de Excelencia SOMMA del RECIDI: severo ochoa y maria maeztu

In [ ]:
import pandas as pd
from src.scrapers.powerbi.queries import recidi_somma_severo_ochoa,recidi_somma_maria_maeztu,get_json_from_powerbi
from src.scrapers.powerbi.powerbi_parser import pbi_json_to_df
from src.shared.utils import slugify, normalize

data_severo_ochoa = get_json_from_powerbi(recidi_somma_severo_ochoa())
data_maria_maeztu = get_json_from_powerbi(recidi_somma_maria_maeztu())

data_severo_ochoa_df = pbi_json_to_df(data_severo_ochoa)
data_maria_maeztu_df = pbi_json_to_df(data_maria_maeztu)

print(data_severo_ochoa_df.columns.tolist())
print(data_maria_maeztu_df.columns.tolist())

centros_somma_df = pd.concat([data_severo_ochoa_df, data_maria_maeztu_df], axis=0)

rename_dict = {
    'ENLACE WEB': 'web',
    'Nombre_Entidad_Mostrar': 'nombre'
}

centros_somma_df = centros_somma_df.rename(columns=rename_dict)
centros_somma_df.columns = [slugify(normalize(col.lower())) for col in centros_somma_df.columns]

centros_somma_df.head()

In [ ]:
from src.geocoders.google_maps import GoogleMaps as GMGeocoder
centros_somma_df['latitud'], centros_somma_df['longitud'], centros_somma_df['direccion_google_maps'] = zip(*centros_somma_df.apply(lambda row: GMGeocoder.geocode(f"{row['nombre']}, {row['provincia']}, España", save_json=True, filename=f"{row['nombre'].replace(' ', '_')}_geocoded.json"), axis=1))

In [ ]:
centros_somma_df.head()

In [ ]:
centros_somma_df.to_csv(
    "../data/silver/centros_somma_recidi.csv", sep=";", index=False)

# Centros tecnológicos del RECIDI

In [ ]:
! ls ../data/bronze/ciencia.gob.es/

In [ ]:
import pandas as pd
data_df = pd.read_excel(
    "../data/bronze/ciencia.gob.es/lista_centros_tecnologicos.xlsx",
    dtype=str,
)
data_df.head()

In [ ]:
centros_tecnologicos_df = data_df.copy()
centros_tecnologicos_df.columns = [
    slugify(normalize(col.lower())) for col in centros_tecnologicos_df.columns]
centros_tecnologicos_df['latitud'], centros_tecnologicos_df['longitud'], centros_tecnologicos_df['direccion_google_maps'] = zip(*centros_tecnologicos_df.apply(
    lambda row: GMGeocoder.geocode(f"{row['acronimo']}, {row['direccion']}, {row['cp']}", save_json=True, filename=f"{row['acronimo'].replace(' ', '_')}_geocoded.json"), axis=1))

In [ ]:
centros_tecnologicos_df.head()

In [ ]:
centros_tecnologicos_df.to_csv(
    "../data/silver/centros_tecnologicos_y_de_apoyo_recidi.csv", sep=";", index=False)

# ICTS del RECIDI


Se puede obtener los datos concretos ya geocodificados a través de https://datos.gob.es/es/catalogo/e05071301-mapa-de-las-infraestructuras-cientifico-tecnicas-singulares-icts

In [ ]:
import pandas as pd
from src.shared.utils import slugify, normalize
data_df = pd.read_csv(
    "../data/bronze/ciencia.gob.es/ReutilizacionDatos_MapaICTS.csv", sep=";", encoding="utf-8", dtype=str
)
data_df.columns = [slugify(normalize(col.lower())) for col in data_df.columns]
data_df

In [ ]:
icts_df = data_df[['icts','web_icts','tipo','instalacion','web_instalacion','area_tematica','entidad_titular_gestora','code','direccion','ccaa','latitud','longitud']].copy()
icts_df['direccion'] = icts_df['direccion'].str.replace(r'\n+', '. ', regex=True).str.strip()
icts_df.dropna(inplace=True)

In [ ]:
icts_df.to_csv(
    "../data/silver/icts.csv", sep=";", index=False
)

# Atributos territoriales

## Datos de Cobertura de Banda Ancha

analizaremos a nivel municpal el % de cobertura de FTTH, velocidad máxima en codniciones de demanda a 1 Gbps, 5G a 3,5 Ghz. en resumen se trata de fibra óptica y 5G puro sin uso auxiliar de redes 4G

In [ ]:
!ls ../data/bronze/avance.digital.gob.es/

vamos a renombrar algunas columnas

In [ ]:
from src.processing.seteleco_parser import process_cobertura_excel


In [ ]:
process_cobertura_excel(
    input_path="../data/bronze/avance.digital.gob.es/Cobertura_BA_España_2021-2024_MUN_PROV_CCAA_Nacional_datosgob.xlsx",
    output_path="../data/silver/cobertura_banda_ancha_municipios_2021_2024.csv",
    sheet_name="Municipio_%viviendas",
    dtype={'CMUN':str}
)
process_cobertura_excel(
    input_path="../data/bronze/avance.digital.gob.es/Cobertura_BA_España_2021-2024_MUN_PROV_CCAA_Nacional_datosgob.xlsx",
    output_path="../data/silver/cobertura_banda_ancha_provincias_2021_2024.csv",
    sheet_name="Provincia_%viviendas",
    dtype=None
)
process_cobertura_excel(
    input_path="../data/bronze/avance.digital.gob.es/Cobertura_BA_España_2021-2024_MUN_PROV_CCAA_Nacional_datosgob.xlsx",
    output_path="../data/silver/cobertura_banda_ancha_ccaa_2021_2024.csv",
    sheet_name="CCAA_%viviendas",
    dtype=None
)

In [ ]:
process_cobertura_excel(
    input_path="../data/bronze/avance.digital.gob.es/Cobertura_BA_España_2021-2024_MUN_PROV_CCAA_Nacional_datosgob.xlsx",
    output_path="../data/silver/cobertura_banda_ancha_nacional_2021_2024.csv",
    sheet_name="Nacional_%viviendas",
    dtype=None
)

## Datos de generación del talento universitario

### Egresados por grado

In [ ]:
!ls ../data/bronze/SIIU

In [ ]:
# fuente
# https://estadisticas.ciencia.gob.es/jaxiPx/Tabla.htm?path=/Universitaria/Alumnado/EEU_2025/GradoCiclo/Egresados/l0/&file=3_3_Egr_Sex_Edad2_Amb_Univ.px&L=0
data_df = pd.read_csv(
    "../data/bronze/ciencia.gob.es/SIIU/3_3_Egr_Sex_Edad2_Amb_Univ.csv",sep=";", encoding="latin-1", dtype=str
)

In [25]:
data_df.head(10)

,Universidad,Ámbito de estudio,Sexo,Grupo de edad,Periodo,Total
0,Todas las universidades,Total,Ambos sexos,Total,2015-2016,203.253
1,Todas las universidades,Total,Ambos sexos,Total,2020-2021,207.646
2,Todas las universidades,Total,Ambos sexos,Total,2021-2022,199.048
3,Todas las universidades,Total,Ambos sexos,Total,2022-2023,201.759
4,Todas las universidades,Total,Ambos sexos,Total,2023-2024,204.800
5,Todas las universidades,Total Educación,Ambos sexos,Total,2015-2016,33.555
6,Todas las universidades,Total Educación,Ambos sexos,Total,2020-2021,33.468
7,Todas las universidades,Total Educación,Ambos sexos,Total,2021-2022,31.276
8,Todas las universidades,Total Educación,Ambos sexos,Total,2022-2023,33.602
9,Todas las universidades,Total Educación,Ambos sexos,Total,2023-2024,32.899


Vamos a filtrar por el período de 2021 a 2024 y limpiar puntos decimales de los totales de egresados

In [26]:
data_df['año'] = data_df['Periodo'].apply(lambda x: x.split('-')[1])
grado_df = data_df[data_df['año'].isin(['2021','2022','2023', '2024'])].copy().reset_index(drop=True)
grado_df['Total'] = pd.to_numeric(grado_df['Total'].str.replace('.',''), errors='coerce').fillna(0).astype(int)

In [ ]:
universidades_df = pd.read_csv(
    "../data/silver/universidades_arc_gis.csv", sep=";", encoding="utf-8", dtype=str
)
universidad_codigo_dict = pd.Series(
    universidades_df['codigo'].values, index=universidades_df['universidad']
).to_dict()
codigo_universidad_dict = pd.Series(
    universidades_df['universidad'].values, index=universidades_df['codigo']
).to_dict()


Aplicamos fuerza bruta para normalizar universidades del dataset con el código de universidad oficial.

In [28]:
from src.shared.utils import fuzzy_match_dict


grado_df['codigo_universidad'] = grado_df['Universidad'].apply(lambda x: fuzzy_match_dict(
    key=x, candidates=universidad_codigo_dict, key_modifiers=['Universidad ', 'Universitat ', 'Universidad de ', 'Universitat de ']))

In [29]:
grado_df[grado_df['codigo_universidad'].isna()]['Universidad'].unique().tolist()

['Todas las universidades',
 'Universidades Públicas',
 'Universidades Públicas Presenciales',
 'Universidades Públicas No Presenciales',
 'Universidades Privadas',
 'Universidades Privadas Presenciales',
 'Universidades Privadas No Presenciales']

verificación de la función de smartmatch

In [30]:
universidades_grado_df = grado_df[['Universidad','codigo_universidad']].drop_duplicates()

for index, row in universidades_grado_df[~universidades_grado_df['codigo_universidad'].isna()].iterrows():
    print(f"resultado del match para '{row['Universidad']}': {codigo_universidad_dict.get(row['codigo_universidad'], 'No encontrado')} (código: {row['codigo_universidad']})")

resultado del match para 'A Coruña': Universidad de A Coruña (código: 037)
resultado del match para 'Alcalá': Universidad de Alcalá (código: 029)
resultado del match para 'Alicante': Universidad de Alicante (código: 001)
resultado del match para 'Almería': Universidad de Almería (código: 048)
resultado del match para 'Autónoma de Barcelona': Universidad Autónoma de Barcelona (código: 022)
resultado del match para 'Autónoma de Madrid': Universidad Autónoma de Madrid (código: 023)
resultado del match para 'Barcelona': Universidad de Barcelona (código: 004)
resultado del match para 'Burgos': Universidad de Burgos (código: 051)
resultado del match para 'Cantabria': Universidad de Cantabria (código: 016)
resultado del match para 'Carlos III de Madrid': Universidad Carlos III de Madrid (código: 036)
resultado del match para 'Castilla-La Mancha': Universidad de Castilla-La Mancha (código: 034)
resultado del match para 'Complutense de Madrid': Universidad Complutense de Madrid (código: 010)
re

In [ ]:
from src.shared.utils import normalize, slugify
grado_df.columns = [slugify(normalize(col.lower())) for col in grado_df.columns]
grado_df.to_csv(
    "../data/silver/egresados_grado_universidades_2021_2024.csv", sep=";", index=False)

### Egresados por Master

In [ ]:
import pandas as pd
from src.shared.utils import normalize, slugify, fuzzy_match_dict
data_df = pd.read_csv(
    "../data/bronze/ciencia.gob.es/SIIU/3_3_Egr_Master_Sex_Edad2_Amb_Univ.csv",sep=";", encoding="latin-1", dtype=str
)
data_df['año'] = data_df['Periodo'].apply(lambda x: x.split('-')[1])
master_df = data_df[data_df['año'].isin(['2021','2022','2023', '2024'])].copy().reset_index(drop=True)
master_df['Total'] = pd.to_numeric(master_df['Total'].str.replace('.',''), errors='coerce').fillna(0).astype(int)
master_df['codigo_universidad'] = master_df['Universidad'].apply(lambda x: fuzzy_match_dict(
    key=x, candidates=universidad_codigo_dict, key_modifiers=['Universidad ', 'Universitat ', 'Universidad de ', 'Universitat de ']))

print("universidades sin normalizar")
for uni in master_df[master_df['codigo_universidad'].isna()]['Universidad'].unique().tolist():
    print(f"- {uni}")
master_df.columns = [slugify(normalize(col.lower())) for col in master_df.columns]
master_df.to_csv(
    "../data/silver/egresados_master_universidades_2021_2024.csv", sep=";", index=False)


universidades sin normalizar
- Todas las universidades
- Universidades Públicas
- Universidades Públicas Presenciales
- Universidades Públicas No Presenciales
- Universidades Públicas Especiales
- Universidades Privadas
- Universidades Privadas Presenciales
- Universidades Privadas No Presenciales


### Egresados por Doctorado


In [ ]:

data_df = pd.read_csv(
    "../data/bronze/ciencia.gob.es/SIIU/3_3_Egr_Doctorado_Sex_Edad2_Amb_Univ.csv",sep=";", encoding="latin-1", dtype=str
)
data_df['año'] = data_df['Periodo'].apply(lambda x: x.split('-')[1])
doctorado_df = data_df[data_df['año'].isin(['2021','2022','2023', '2024'])].copy().reset_index(drop=True)
doctorado_df['Total'] = pd.to_numeric(doctorado_df['Total'].str.replace('.',''), errors='coerce').fillna(0).astype(int)
doctorado_df['codigo_universidad'] = doctorado_df['Universidad'].apply(lambda x: fuzzy_match_dict(
    key=x, candidates=universidad_codigo_dict, key_modifiers=['Universidad ', 'Universitat ', 'Universidad de ', 'Universitat de ']))

print("universidades sin normalizar")
for uni in doctorado_df[doctorado_df['codigo_universidad'].isna()]['Universidad'].unique().tolist():
    print(f"- {uni}")
doctorado_df.columns = [slugify(normalize(col.lower())) for col in doctorado_df.columns]
doctorado_df.to_csv(
    "../data/silver/egresados_doctorado_universidades_2021_2024.csv", sep=";", index=False)

universidades sin normalizar
- Todas las universidades
- Universidades Públicas
- Universidades Públicas Presenciales
- Universidades Públicas No Presenciales
- Universidades Públicas Especiales
- Universidades Privadas
- Universidades Privadas Presenciales
- Universidades Privadas No Presenciales


## Afiliación SS
Datos obtenidos del portal de estadísticas de la Seguridad Social.

In [ ]:
!ls ../data/bronze/SS/afiliacion/

In [1]:
import pandas as pd
from src.shared.utils import normalize, slugify
data_df = pd.read_excel(
    "../data/bronze/SS/afiliacion/MUNCNAE1221.xlsx", dtype=str
)
data_df.columns = [slugify(normalize(col.lower())) for col in data_df.columns]
data_df.head(1)

,fecha,cod_provincia,provincia,cod_municipio,municipio,cod_regimen,regimen,cod_cnae,cnae,cod_sexo,sexo,afiliados
0,202112,1,ARABA/ÁLAVA,1000,NO CONSTA MUNICIPIO,111,REGIMEN GENERAL,NaN,NO CONSTA,2,MUJER,<5


Sólo nos interesa conocer la afiliación en Regimen general(111) y Regimen por cuenta propia/autónomos (521)

In [3]:
from src.processing.seg_social_parser import process_ss_afiliation_excel

In [ ]:
afiliacion_excels = [
    "../data/bronze/SS/afiliacion/MUNCNAE1220.xlsx",
    "../data/bronze/SS/afiliacion/MUNCNAE1221.xlsx",
    "../data/bronze/SS/afiliacion/MUNCNAE1222.xlsx",
    "../data/bronze/SS/afiliacion/MUNCNAE1223.xlsx",
    "../data/bronze/SS/afiliacion/MUNCNAE1224.xlsx",
]

municipios_df = pd.read_csv(
    "../data/silver/municipios.csv", sep=",", encoding="utf-8", dtype=str
)
municipios_df.head(10)
municipios_df['cod_municipio'] = municipios_df['cpro'] + municipios_df['cmun']
municipios_dict = pd.Series(municipios_df.nombre.values, index=municipios_df.cod_municipio).to_dict()

for excel in afiliacion_excels:
    df = process_ss_afiliation_excel(excel, municipios_dict, export_to_csv=True)
    print(f"Archivo procesado: {excel}, filas resultantes: {len(df)}")

## precio del alquiler
Datos del portal del ministerio de vivienda.

In [ ]:
!ls ../data/bronze/mivau.gob.es/

In [ ]:

import pandas as pd
from src.shared.utils import normalize, slugify
def exporter(file_path,export_to_csv, export_to_parquet, df):
    export_to_parquet = not export_to_csv and export_to_parquet
    if export_to_csv:
        df.to_csv(
            file_path+".csv", sep=";", index=False
        )
    if export_to_parquet:
        df.to_parquet(
            file_path+".parquet", index=False, compression='snappy', engine='pyarrow'
        )
def process_alquiler_excel(file_path: str,
                            years:list,
                            sheet_name:str="Municipios", 
                            export_to_csv=False,
                            export_to_parquet=True,
                            single_export_file=False, 
                            export_directory: str = "../data/silver/") -> pd.DataFrame:
    def get_rename_dict(cols):
        rename_dict = {}
        for col in cols:
            rename_dict[col] = col[:-3] if col[-2:].isdigit() else col
        return rename_dict

    
    data_df = pd.read_excel(
        file_path, sheet_name=sheet_name, dtype=str
    ).rename(columns=lambda x: slugify(normalize(x.lower()))).rename(columns={
        'cumun': 'cmun',
        'cpro': 'cpro'
    })
    KIND = ['vu', 'vc'] # vivienda unifamiliar y vivienda colectiva
    LEVEL = ['25','m','75'] # percentiles 25, mediana y 75
    templates = ["alqm2_lv_{level}_{kind}_", "alqtbid12_{level}_{kind}_"]
    price_cols_prefixes = [template.format(level=level, kind=kind) for template in templates for kind in KIND for level in LEVEL]
    id_cols = [col for col in ['cmun', 'cpro', 'ccaa','litccaa'] if col in data_df.columns]
    dfs = []
    for year in years:
        year_2_digits = str(year)[-2:]
        price_cols = [col+year_2_digits for col in price_cols_prefixes]
        rename_dict = get_rename_dict(price_cols)
        
        alq_df = data_df[id_cols + price_cols].copy().rename(columns=rename_dict)
        alq_df['anno'] = year
        alq_df.dropna(subset=[id_cols[0]], inplace=True)
        alq_df.drop_duplicates(inplace=True)
        
        if single_export_file:
            dfs.append(alq_df)
            continue
        
        exporter(
            f"{export_directory}/alquiler_{slugify(normalize(sheet_name.lower()))}_{year}",
            export_to_csv,
            export_to_parquet,
            alq_df
        )
    if single_export_file and dfs:
        df = pd.concat(dfs, axis=0)
        
        exporter(
            f"{export_directory}/alquiler_{slugify(normalize(sheet_name.lower()))}_{years[0]}_{years[-1]}",
            export_to_csv,
            export_to_parquet,
            df
        )
    return data_df

In [ ]:
#process_alquiler_excel("../data/bronze/mivau.gob.es/2025-09-10_bd_SERPAVI_2011-2023.xlsx",
#                       sheet_name="Municipios", years=[2021, 2022, 2023], export_to_csv=True)
#process_alquiler_excel("../data/bronze/mivau.gob.es/2025-09-10_bd_SERPAVI_2011-2023.xlsx",
#                       sheet_name="Provincias", years=[2021, 2022, 2023], export_to_csv=True)
df = process_alquiler_excel("../data/bronze/mivau.gob.es/2025-09-10_bd_SERPAVI_2011-2023.xlsx",
                          sheet_name="CCAA", years=[2021, 2022, 2023], export_to_parquet=True, single_export_file=True)

In [ ]:
alquiler_df = pd.read_parquet("../data/silver/alquiler_municipios_2021_2023.parquet")
alquiler_df.query("cpro == '03'")

## Densidad poblacional

Procesamiento de las hojas de calculo del padrón municipal.

In [ ]:
import pandas as pd
from src.shared.utils import normalize, slugify
def process_population_excel(
        file_path:str,
        year:str = None,
        export_to_csv=False,
        export_to_parquet=True,
        export_directory: str = "../data/silver/"
):
    data_df = pd.read_excel(
        file_path, dtype=str, skiprows=1
    )

    data_df.columns = [slugify(normalize(col.lower())) for col in data_df.columns]

    data_df.rename(columns={
        'pob': 'poblacion'
    }, inplace=True)

    pob_col = data_df.columns[data_df.columns.str.contains('pob')][0]
    data_df.rename(columns={pob_col: 'poblacion'}, inplace=True)
    
    data_df['cmun'] = data_df['cpro'] + data_df['cmun'].astype(str).str.zfill(3)
    if year is None:
        year = "20" + pob_col[-2:]
    data_df['anno'] = year
    data_df['poblacion'] = data_df['poblacion'].str.replace('.','').astype(int)
    data_df['hombres'] = data_df['hombres'].str.replace('.','').astype(int)
    data_df['mujeres'] = data_df['mujeres'].str.replace('.','').astype(int)
    exporter(
        f"{export_directory}/poblacion_{year}",
        export_to_csv,
        export_to_parquet,
        data_df
    )
    return data_df

In [ ]:
process_population_excel(
    "../data/bronze/INE/pobmun/pobmun20.xlsx", year="2020", export_to_parquet=True, export_directory="../data/silver/"
)
process_population_excel(
    "../data/bronze/INE/pobmun/pobmun21.xlsx", year="2021", export_to_parquet=True, export_directory="../data/silver/"
)
process_population_excel(
    "../data/bronze/INE/pobmun/pobmun22.xlsx", year="2022", export_to_parquet=True, export_directory="../data/silver/"
)
process_population_excel(
    "../data/bronze/INE/pobmun/pobmun23.xlsx", year="2023", export_to_parquet=True, export_directory="../data/silver/"
)
process_population_excel(
    "../data/bronze/INE/pobmun/pobmun24.xlsx", year="2024", export_to_parquet=True, export_directory="../data/silver/"
)
process_population_excel(
    "../data/bronze/INE/pobmun/pobmun25.xlsx", year="2025", export_to_parquet=True, export_directory="../data/silver/"
)

,cpro,provincia,cmun,nombre,poblacion,hombres,mujeres,anno
0,01,Araba/Álava,01001,Alegría-Dulantzi,2955,1522,1433,2025
1,01,Araba/Álava,01002,Amurrio,10364,5133,5231,2025
2,01,Araba/Álava,01003,Aramaio,1354,687,667,2025
3,01,Araba/Álava,01004,Artziniega,1868,923,945,2025
4,01,Araba/Álava,01006,Armiñón,236,124,112,2025
...,...,...,...,...,...,...,...,...
8127,50,Zaragoza,50901,Biel,166,99,67,2025
8128,50,Zaragoza,50902,Marracos,82,42,40,2025
8129,50,Zaragoza,50903,Villamayor de Gállego,2841,1445,1396,2025
8130,51,Ceuta,51001,Ceuta,83595,42153,41442,2025


# Datos aeropuertos

A través de los datos del ministerio de fomento se puede obtener datos del tráfico de los aeropuertos españoles, incluyendo el número de asientos y operaciones por aeropuerto, clase de tráfico y año.

In [ ]:
import pandas as pd
data_df = pd.read_csv("../data/bronze/apps.fomento.gob.es/CO430_Num_Asientos_Opers__AeropBase_CiasAereasMasRepresen_Anyo.csv", dtype={'asientos': str, 'operaciones': str}, sep=",")

In [2]:
data_df['asientos'] = data_df['asientos'].fillna('0').str.replace('.','').astype(int)
data_df['operaciones'] = data_df['operaciones'].fillna('0').str.replace('.','').astype(int)
vuelos_df = data_df.groupby(['year','aeropuerto_base','Clase_trafico'])[['asientos','operaciones']].sum()

In [3]:
vuelos_df = data_df.groupby(['year','aeropuerto_base','Clase_trafico'])[['asientos','operaciones']].sum().reset_index()
vuelos_df['codigo_aeropuerto'] = vuelos_df['aeropuerto_base'].apply(lambda x: x.split(':')[0].strip())
vuelos_df['nombre_aeropuerto'] = vuelos_df['aeropuerto_base'].apply(lambda x: x.split(':')[1].strip() if ':' in x else x.strip())

In [ ]:
from src.geocoders.google_maps import GoogleMaps as GMGeocoder
def geocode_aeropuerto(airport_name):
    term = f"{airport_name} Airport, Spain"
    return GMGeocoder.geocode(term, save_json=True, filename=f"{airport_name}_geocoded.json")

aeropuertos_unicos = vuelos_df['aeropuerto_base'].unique()
aeropuerto_dict = {aeropuerto: geocode_aeropuerto(aeropuerto) for aeropuerto in aeropuertos_unicos}

vuelos_df['latitud'], vuelos_df['longitud'], vuelos_df['direccion_google_maps'] = zip(*vuelos_df.apply(lambda row: aeropuerto_dict.get(row['aeropuerto_base'], (None, None, None)), axis=1))

In [ ]:
import re
vuelos_df['codigo_postal'] = vuelos_df.apply(lambda row: re.search(r'(\d{5})', row['direccion_google_maps']).group(1) if re.search(r'(\d{5})', row['direccion_google_maps']) else None, axis=1)

In [ ]:
vuelos_df[vuelos_df['codigo_postal'].isna()]['direccion_google_maps'].unique().tolist()

In [ ]:
vuelos_df.loc[vuelos_df['direccion_google_maps']=='César Manrique-Lanzarote Arpt (ACE), Las Palmas, Spain', 'codigo_postal'] = '35509'

In [ ]:
from src.geocoders.CorreosApi import CorreosApi
correos_geocoder = CorreosApi()

vuelos_df['cmun'] = vuelos_df.apply(lambda row: correos_geocoder.get_municipality_code_from_postal_code(row['codigo_postal']) if pd.notnull(row['codigo_postal']) else None, axis=1)

In [ ]:
vuelos_df[vuelos_df['cmun'].isna()]['codigo_postal'].unique().tolist()

In [ ]:
provincias_df = pd.read_csv("../data/silver/ine_ccaa_y_provincias.csv", sep=";", encoding="utf-8", dtype=str)
codigo_provincia_a_ccaa_dict = provincias_df.set_index('cpro')['codauto'].to_dict()
vuelos_df['cpro'] = vuelos_df['cmun'].apply(lambda x: x[:2] if pd.notnull(x) else None)
vuelos_df['cpro'] = vuelos_df.apply(lambda row: row['codigo_postal'][:2] if row['cpro'] is None else row['cpro'], axis=1)
vuelos_df['ccaa'] = vuelos_df['cpro'].map(codigo_provincia_a_ccaa_dict)

In [ ]:
vuelos_df.drop(columns=['aeropuerto_base'], inplace=True)

In [ ]:
municipios_df = pd.read_csv("../data/silver/municipios.csv", sep=",", encoding="utf-8", dtype=str)
municipios_df['cmun'] = municipios_df['cpro'] + municipios_df['cmun']
municipios_dict = municipios_df.set_index('nombre')['cmun'].to_dict()
mask = vuelos_df['cmun'].isna()

extracted = vuelos_df.loc[mask, 'direccion_google_maps'] \
    .str.extract(r",\s*\d+\s+([A-Za-zÀ-ÿ' ]+),")[0]

vuelos_df.loc[mask, 'cmun'] = extracted.map(municipios_dict)

In [8]:
mask = vuelos_df.query('cmun.isna() and direccion_google_maps.str.contains("Rutis")').index

vuelos_df.loc[mask, 'cmun'] = municipios_dict.get('Culleredo', None)

mask = vuelos_df.query('cmun.isna() and direccion_google_maps.str.contains("Santiago del Monte")').index

vuelos_df.loc[mask, 'cmun'] = municipios_dict.get('Castrillón', None)

In [9]:

vuelos_df.query("cmun.isna()")['direccion_google_maps'].unique().tolist()

[]

In [11]:
from src.shared.utils import normalize, slugify
vuelos_df.columns = [slugify(normalize(col.lower())) for col in vuelos_df.columns]

In [ ]:
vuelos_df.to_csv("../data/silver/vuelos_aeropuertos_clase_trafico.csv", sep=";", index=False)